# Using Middleware to add HITL in the Agentic Process

In [1]:
from dotenv import load_dotenv

# Remember: the .env file in the project folder must have the line
# OPENAI_API_KEY=sk-... (no quotes, no spaces).
load_dotenv()

True

#### Update note: current OpenAI models

* The original example is very good for learning the concept, and it worked with the OpenAI models available when the course was recorded: the agent used `gpt-4o-mini`.
* OpenAI has deprecated `gpt-4o-mini` (shutdown on January 14, 2027) and recommends `gpt-6-luna` as its replacement. We change it now so that the exercise keeps working after that date.
* This agent uses tools (`read_email` and `send_email`). `gpt-6-luna` is a reasoning model. Through the Chat Completions endpoint, which LangChain uses by default, it only accepts tools when reasoning is turned off. That is why we create the model with `init_chat_model(MODEL, reasoning_effort="none")` and pass that model object to `create_agent`, instead of the model name as text. The model then answers directly, like `gpt-4o-mini` did. (The other option OpenAI offers is the Responses API: `init_chat_model(MODEL, use_responses_api=True)`.)
* The human-in-the-loop part does not change: the middleware still pauses before `send_email`, and you still answer with approve, reject or edit.
* The reply email the agent writes will be worded differently from the one in the video.
* The explanations below keep the original code (`model="gpt-4o-mini"`). The terminal version shown further down also keeps the original code; the file `013-mid-to-add-HITL.py` itself is already updated to `gpt-6-luna`, so run that file rather than copying the code from the text.
* Check available models at developers.openai.com/api/docs/deprecations, and the current human-in-the-loop options at docs.langchain.com.

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

from langchain.agents import create_agent, AgentState
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from pprint import pprint

class EmailState(AgentState):
    email: str

# Original version: this example used gpt-4o-mini.
# OpenAI has deprecated it (shutdown on January 14, 2027) and recommends
# gpt-6-luna as its replacement: small, fast and affordable for practice.
MODEL = "gpt-6-luna"

# This agent uses tools (read_email and send_email). Through the Chat
# Completions endpoint, which LangChain uses by default, gpt-6-luna only
# accepts tools when reasoning is turned off, so we set reasoning_effort="none".
# The model then answers directly, like gpt-4o-mini did. (The other option
# OpenAI offers is the Responses API: init_chat_model(MODEL, use_responses_api=True).)
# We create the model object here, instead of passing its name as text,
# so that we can add this parameter.
model = init_chat_model(MODEL, reasoning_effort="none")

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)


from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response.")],
        "email": "Hi, Donald. I will be in town tomorrow, will you have an available room in the White House? Best, Julio."
    },
    config=config
)

print(response['__interrupt__'])


from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "Sorry, we have Vlad Putin tomorrow for dinner and the WH will be full. Best, Donald."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "Sure! You will be our guest of honor. Best, Donald."},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

## A lot of things going on there! Let's explain the previous code in simple terms
Below is a **simple, line-by-line** explanation of what your code is doing, focused on **Middleware + Human-in-the-Loop (HITL)** for a beginner.

Key idea first: **the agent can “think and choose tools,” but HITL middleware can “pause” right before certain tools run, so a human can approve / reject / edit the action.** 

---

#### 1) Import the tool helpers

```python
from langchain.tools import tool, ToolRuntime
```

* `tool` = a decorator that turns a normal Python function into a **LangChain tool** the agent can call.
* `ToolRuntime` = an object LangChain passes into tools (when you ask for it) that can include things like **agent state** and runtime context.

---

#### 2) Tool #1: read_email (reads from agent state)

```python
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]
```

Line-by-line:

* `@tool` → “Make this function available as a tool.”
* `def read_email(runtime: ToolRuntime) -> str:`
  This tool receives a `runtime` object (injected by LangChain).
* `return runtime.state["email"]`
  The tool looks inside the **agent’s state** and returns the `"email"` field.

So: **this tool doesn’t fetch real email**—it just reads a value stored in state.

(Using custom state fields like this is exactly what `state_schema` is for.)

---

#### 3) Tool #2: send_email (pretends to send)

```python
@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"
```

Line-by-line:

* `@tool` → make it callable by the agent.
* `body: str` → the agent must provide a text body.
* returns `"Email sent"` → it’s a stub (fake). In real life, this would call Gmail/SMTP/etc.

---

#### 4) Import agent + HITL + persistence

```python
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from pprint import pprint
```

* `create_agent` = builds a ready-to-run agent that loops: **think → tool call → observe → think → …**
* `AgentState` = base class for the agent’s short-term memory/state (like `messages`).
* `InMemorySaver` = a “checkpointer” that saves the agent state so it can **pause and resume** later (needed for interrupts).
* `HumanInTheLoopMiddleware` = middleware that can interrupt tool calls and require human decisions.
* `pprint` = pretty-print Python objects.

---

#### 5) Define your custom agent state (adds `email`)

```python
class EmailState(AgentState):
    email: str
```

* This means the agent’s state will include everything from `AgentState` (like `messages`) **plus** an `email` field.
* The goal: tools (like `read_email`) can read `state["email"]`.

---

#### 6) Create the agent + configure HITL policy

```python
agent = create_agent(
    model="gpt-4o-mini",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)
```

Line-by-line (inside `create_agent`):

* `model="gpt-4o-mini"` → which LLM to use.
* `tools=[read_email, send_email]` → the only actions the agent can take.
* `state_schema=EmailState` → adds the `email` field into the agent’s state.
* `checkpointer=InMemorySaver()` → required because HITL can **pause**, and you need saved state to **resume**.
* `middleware=[HumanInTheLoopMiddleware(...)]` → install the HITL “gatekeeper.”

Now the HITL config:

```python
interrupt_on={
    "read_email": False,
    "send_email": True,
},
```

* `"read_email": False` → let the agent read without asking you.
* `"send_email": True` → **pause before actually sending** and ask a human what to do.

This is the core HITL idea: **allow safe tools automatically, but protect risky tools** (like “send an email”).

Also:

```python
description_prefix="Tool execution requires approval"
```

* Adds a friendly label/message to the interrupt prompt shown to the human.

---

#### 7) Make an initial request to the agent

```python
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}
```

* `HumanMessage` represents the user’s message.
* `thread_id="1"` is crucial: it tells LangGraph/LangChain **“this is the same ongoing conversation”**, so you can resume after an interrupt using the same thread. (Persistence + thread identity is how “pause/resume” works.)

Now invoke:

```python
response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response.")],
        "email": "Hi, Donald. I will be in town tomorrow, will you have an available room in the White House? Best, Julio."
    },
    config=config
)
```

What this input means:

* `"messages": [...]` → the conversation history starts with your request.
* `"email": "Hi, Donald...."` → you’re placing an email into the agent’s **state** (because your `EmailState` allows it).

So the agent can:

1. call `read_email` (allowed)
2. decide on a reply
3. try to call `send_email` → **this triggers HITL pause** (because interrupt_on says True)

---

#### 8) Print the interrupt payload

```python
print(response['__interrupt__'])
```

* When HITL middleware pauses execution, the returned response contains a `__interrupt__` field describing:

  * which tool call is about to run,
  * what arguments it wants to use,
  * and what decisions the human can provide.

LangChain’s docs explicitly describe this “interrupted result has `__interrupt__`” behavior.

---

#### 9) Resume #1: approve the tool call

```python
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)
```

Line-by-line:

* `Command(resume=...)` is how you “continue from the pause” by providing the human’s decision.
* `decisions: [{"type": "approve"}]` means:

  * “Run the tool exactly as the agent proposed.”
* Same `config` with same `thread_id` → resumes the paused run.

So: **the email gets “sent” (in your fake tool), and the agent continues.** 

---

#### 10) Resume #2: reject, with feedback message

```python
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "Sorry, we have Vlad Putin tomorrow for dinner and the WH will be full. Best, Donald."
                }
            ]
        }
    ), 
    config=config
)   

pprint(response)
```

What this does:

* `"type": "reject"` → “Do NOT run the tool.”
* `"message": ...` → feedback for the agent (like “here’s what to say instead / why we’re not doing it”).

In HITL terms: reject = “skip this action, and here’s human guidance.”


---

#### 11) Resume #3: edit the tool call (change arguments)

```python
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        "name": "send_email",
                        "args": {"body": "Sure! You will be our guest of honor. Best, Donald."},
                    }
                }
            ]
        }
    ), 
    config=config
)   

pprint(response)
```

What this does:

* `"type": "edit"` → “Run the tool, but NOT with the agent’s original arguments.”
* `edited_action` lets the human specify:

  * the tool name (`send_email`)
  * the new args (`body=...`)

So: the agent wanted to send something, but **you override the content** before it “sends.”

---

#### Mental model: what happens end-to-end

1. You call `agent.invoke(...)`.
2. The agent thinks and calls tools.
3. HITL middleware checks each tool call:

   * `read_email` → allowed automatically
   * `send_email` → **interrupt**
4. You inspect `response["__interrupt__"]`.
5. You resume with a `Command(resume=...)` decision:

   * `approve` → run as-is
   * `reject` → skip + feedback
   * `edit` → run with changed args ([docs.langchain.com][1])

That’s it: **Middleware = control layer; HITL middleware = “human approval gate” for risky tools.**

## Well, that is an interesting code, but if we wanted to try this app providing actual human feedback, how should the code be?

To make this work with **actual human feedback**, you need to separate the execution into phases and add a way to capture real user input. Here's how to modify the code:

## First, below you have the code version to run in your Jupyter Notebook (this will NOT work if you try to run it in Visual Studio Code)

#### Update note: current OpenAI models

* This cell creates its own agent, so it has the same change as the first example: `gpt-6-luna` instead of `gpt-4o-mini`, created with `init_chat_model(MODEL, reasoning_effort="none")` because the agent uses tools.
* The rest of the cell (the interrupt loop and your decisions) does not change.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent, AgentState
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
from langgraph.types import Command
from pprint import pprint
from IPython.display import display, Markdown

# Tool definitions
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    return f"✅ Email sent successfully!\n\nBody: {body}"

# State and agent setup
class EmailState(AgentState):
    email: str

# Original version: this example used gpt-4o-mini.
# OpenAI has deprecated it (shutdown on January 14, 2027) and recommends
# gpt-6-luna as its replacement: small, fast and affordable for practice.
MODEL = "gpt-6-luna"

# This agent uses tools (read_email and send_email). Through the Chat
# Completions endpoint, which LangChain uses by default, gpt-6-luna only
# accepts tools when reasoning is turned off, so we set reasoning_effort="none".
# The model then answers directly, like gpt-4o-mini did. (The other option
# OpenAI offers is the Responses API: init_chat_model(MODEL, use_responses_api=True).)
# We create the model object here, instead of passing its name as text,
# so that we can add this parameter.
model = init_chat_model(MODEL, reasoning_effort="none")

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

# Configuration
config = {"configurable": {"thread_id": "1"}}

# PHASE 1: Initial invocation
display(Markdown("## 🚀 PHASE 1: Starting agent..."))

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response.")],
        "email": "Hi, Donald. I will be in town tomorrow, will you have an available room in the White House? Best, Julio."
    },
    config=config
)

# Function to handle interrupts (we might need to handle multiple interrupts)
def handle_interrupt(response, config, interrupt_number=1):
    """Handle a single interrupt and return the updated response"""
    
    if '__interrupt__' not in response:
        return response, False  # No interrupt, we're done
    
    display(Markdown(f"### ⚠️ INTERRUPT #{interrupt_number} - Human approval required!"))
    
    # Extract interrupt info
    interrupt_obj = response['__interrupt__'][0]
    interrupt_value = interrupt_obj.value
    
    # Extract tool information from the interrupt value
    action_requests = interrupt_value.get('action_requests', [])
    if action_requests:
        action_request = action_requests[0]
        tool_name = action_request.get('name', 'unknown')
        tool_args = action_request.get('args', {})
        description = action_request.get('description', '')
        
        display(Markdown(f"**🤖 Agent wants to call:** `{tool_name}`"))
        display(Markdown(f"**📝 With arguments:**"))
        pprint(tool_args)
        
        if description:
            display(Markdown(f"**📄 Description:**"))
            print(description)
    else:
        print("⚠️ No action requests found in interrupt")
        return response, False
    
    # PHASE 2: Get human input
    display(Markdown("---"))
    display(Markdown(f"## 🤔 DECISION TIME (Interrupt #{interrupt_number})"))
    display(Markdown("""
    **What do you want to do?**
    - Type `1` to **Approve** - Let the agent execute as planned
    - Type `2` to **Reject** - Stop the action and provide feedback
    - Type `3` to **Edit** - Modify the content before executing
    """))
    
    choice = input("Enter your choice (1/2/3): ").strip()
    
    # PHASE 3: Resume based on decision
    display(Markdown("---"))
    display(Markdown(f"## ⚡ RESUMING (Interrupt #{interrupt_number})..."))
    
    if choice == "1":
        # APPROVE
        display(Markdown("### ✅ You **APPROVED** the action"))
        new_response = agent.invoke(
            Command(resume={"decisions": [{"type": "approve"}]}),
            config=config
        )
        return new_response, True
        
    elif choice == "2":
        # REJECT
        display(Markdown("### ❌ You **REJECTED** the action"))
        feedback = input("Enter your feedback message: ").strip()
        new_response = agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {
                            "type": "reject",
                            "message": feedback
                        }
                    ]
                }
            ),
            config=config
        )
        return new_response, True
        
    elif choice == "3":
        # EDIT
        display(Markdown("### ✏️ You chose to **EDIT** the action"))
        new_body = input("Enter the new email body: ").strip()
        new_response = agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {
                            "type": "edit",
                            "edited_action": {
                                "name": tool_name,
                                "args": {"body": new_body}
                            }
                        }
                    ]
                }
            ),
            config=config
        )
        return new_response, True
        
    else:
        display(Markdown("### ❌ Invalid choice"))
        return response, False

# Handle all interrupts in a loop
interrupt_count = 0
max_interrupts = 5  # Safety limit to prevent infinite loops

while '__interrupt__' in response and interrupt_count < max_interrupts:
    interrupt_count += 1
    response, continued = handle_interrupt(response, config, interrupt_count)
    
    if not continued:
        break

# FINAL RESULT
display(Markdown("---"))
display(Markdown("## 🎉 FINAL RESULT"))

if '__interrupt__' in response:
    display(Markdown("⚠️ **Still has pending interrupts** (reached max limit or invalid choice)"))
    pprint(response['__interrupt__'])
else:
    display(Markdown("✅ **Process completed successfully!**"))
    
    # Show only the final messages (skip the interrupt details)
    if 'messages' in response:
        display(Markdown("### 📧 Final Messages:"))
        for msg in response['messages'][-3:]:  # Show last 3 messages
            print(f"\n{msg.type.upper()}: {msg.content if hasattr(msg, 'content') else msg}")

## Ok, let's explain the most important parts of the previous code in simple terms

#### **🎯 The Big Picture**

This code creates an AI agent that **pauses before taking sensitive actions** (like sending emails) and waits for your approval. Think of it like having a "send" button that you must click before an email goes out.

---

#### **1. The Middleware Setup - The "Gatekeeper"**

```python
middleware=[
    HumanInTheLoopMiddleware(
        interrupt_on={
            "read_email": False,  # ✅ Auto-approve (no pause)
            "send_email": True,   # ⚠️ PAUSE HERE! (need approval)
        },
    ),
]
```

**What this means:**
- `interrupt_on` is like a security checkpoint
- `False` = "Go ahead, no approval needed"
- `True` = "STOP! Wait for human approval"
- In this case: reading is safe ✅, but sending needs approval ⚠️

**Why it matters:** This one simple dictionary controls which actions are safe vs. dangerous!

---

#### **2. The Checkpointer - The "Save Game" Feature**

```python
checkpointer=InMemorySaver(),
```

**What this means:**
- Like a video game checkpoint that saves your progress
- When the agent pauses, it saves EVERYTHING about the conversation
- You can come back later and resume from exactly where you left off

**Why it matters:** Without this, the agent can't pause and resume. The checkpoint is **mandatory** for HITL to work!

---

#### **3. The Thread ID - Your "Conversation Name"**

```python
config = {"configurable": {"thread_id": "1"}}
```

**What this means:**
- Every conversation needs a unique ID (like "conversation_1", "conversation_2", etc.)
- This tells the agent "save this conversation under this name"
- When you resume, you use the SAME thread_id to continue that exact conversation

**Why it matters:** If you change the thread_id, you start a completely new conversation!

---

#### **4. The Interrupt Response - What Happens When Paused**

```python
if '__interrupt__' in response:
    # Agent is paused and waiting!
```

**What this means:**
- When the agent hits a tool that needs approval, `__interrupt__` appears in the response
- It's like a **red flag** 🚩 saying "I'm waiting for your decision!"
- Inside `__interrupt__` is all the information about what the agent wants to do

**Why it matters:** This is how you KNOW the agent is paused and waiting for you!

---

#### **5. Extracting the Tool Information**

```python
interrupt_obj = response['__interrupt__'][0]
interrupt_value = interrupt_obj.value
action_requests = interrupt_value.get('action_requests', [])
action_request = action_requests[0]
tool_name = action_request.get('name')
tool_args = action_request.get('args')
```

**What this means (step-by-step):**
1. Get the interrupt object (the pause notification)
2. Extract its `value` (the details)
3. Look inside `action_requests` (what the agent wants to do)
4. Get the tool `name` (e.g., "send_email")
5. Get the tool `args` (e.g., the email body)

**Why it matters:** This tells you EXACTLY what the agent is trying to do, so you can make an informed decision!

---

#### **6. The Three Decision Types - Your Choices**

```python
# Option 1: APPROVE
Command(resume={"decisions": [{"type": "approve"}]})

# Option 2: REJECT
Command(resume={"decisions": [{"type": "reject", "message": "..."}]})

# Option 3: EDIT
Command(resume={"decisions": [{"type": "edit", "edited_action": {...}}]})
```

**What each means:**

**APPROVE** ✅
- "Yes, do exactly what you planned"
- The tool executes with the original arguments

**REJECT** ❌
- "No, don't do that"
- You provide feedback explaining why
- The agent receives your message and can try something else

**EDIT** ✏️
- "Do it, but change the details first"
- You modify the arguments (e.g., change the email body)
- The tool executes with YOUR modified version

**Why it matters:** These three options give you complete control over the agent's actions!

---

#### **7. The Resume Command - Continuing After Pause**

```python
agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config  # ← SAME thread_id!
)
```

**What this means:**
- `Command(resume=...)` tells the agent "here's my decision, continue!"
- You MUST use the same `config` (thread_id) to resume the right conversation
- The agent picks up exactly where it left off

**Why it matters:** This is how you "unpause" the agent after making your decision!

---

#### **8. The Interrupt Loop - Handling Multiple Pauses**

```python
while '__interrupt__' in response and interrupt_count < max_interrupts:
    interrupt_count += 1
    response, continued = handle_interrupt(response, config, interrupt_count)
```

**What this means:**
- Sometimes the agent gets interrupted MORE THAN ONCE
- This loop keeps handling interrupts until there are no more
- `max_interrupts` prevents infinite loops (safety!)

**Why it matters:** Without this loop, you'd only handle the first interrupt and miss the rest!

---

#### **🎓 Key Takeaways for Beginners**

1. **Middleware = Security Guard** 
   - Decides which tools need approval

2. **Checkpointer = Save Button**
   - Required for pausing and resuming

3. **Thread ID = Conversation ID**
   - Must stay the same to resume correctly

4. **`__interrupt__` = Red Flag** 🚩
   - Means "I'm paused, need your input!"

5. **Three Decisions = Your Control**
   - Approve, Reject, or Edit the action

6. **Command(resume=...) = Resume Button**
   - Tells the agent to continue with your decision

7. **Loop = Handle All Interrupts**
   - Some actions trigger multiple pauses

---

## **💡 Real-World Analogy**

Think of it like a **mail room assistant**:

1. **Reading mail** (read_email) → They can do this freely ✅
2. **Sending mail** (send_email) → They must show you first and get approval ⚠️
3. **You review it** → You can approve, reject, or edit the letter
4. **They continue** → After your decision, they finish the task

The middleware is the company policy, the checkpointer is their notepad (remembering where they left off), and the thread_id is the specific task they're working on.

## Here are the Jupyter-specific parts of the previous code

#### **1. IPython Display Imports**
```python
from IPython.display import display, Markdown
```
**Why Jupyter-specific:**
- `display()` and `Markdown()` are **only available in Jupyter/IPython**
- They don't exist in regular Python scripts
- They render rich formatted output in notebook cells

**What they do:**
- `Markdown()` - Renders markdown (headers, bold, lists) beautifully in Jupyter
- `display()` - Shows the rendered markdown in the output

---

#### **2. Markdown Formatting for Rich Output**
```python
display(Markdown("## 🚀 PHASE 1: Starting agent..."))
display(Markdown("### ⚠️ INTERRUPT - Human approval required!"))
display(Markdown("**🤖 Agent wants to call:** `send_email`"))
```

**Why Jupyter-specific:**
- In a regular Python script, you'd use `print()` statements
- Markdown makes the output **much prettier** in Jupyter with:
  - Headers (`##`, `###`)
  - Bold text (`**text**`)
  - Code formatting (`` `code` ``)
  - Emojis render nicely

**Regular Python alternative:**
```python
print("=" * 60)
print("PHASE 1: Starting agent...")
print("=" * 60)
```

---

#### **3. Interactive Input Boxes**
```python
choice = input("Enter your choice (1/2/3): ").strip()
new_body = input("Enter the new email body: ").strip()
feedback = input("Enter your feedback message: ").strip()
```

**Why Jupyter-specific behavior:**
- In Jupyter, `input()` creates an **interactive text box** below the cell
- You type directly in the notebook interface
- In a terminal/console, `input()` works differently (command line prompt)

**What it looks like in Jupyter:**
```
Enter your choice (1/2/3): [____text box appears here____]
```

---

#### **4. Single Cell Execution**
```python
# ALL CODE IN ONE CELL ← Important for Jupyter!
```

**Why Jupyter-specific:**
- The **entire workflow must run in ONE cell** for HITL to work properly
- Splitting across multiple cells would lose the state
- The checkpointer and config need to stay in memory throughout

**If you split it:**
```python
# Cell 1
response = agent.invoke(...)  # Creates interrupt

# Cell 2 (DOESN'T WORK!)
response = agent.invoke(Command(resume=...))  # Lost context!
```

---

#### **5. Pretty Printing with pprint**
```python
from pprint import pprint
pprint(response)  # Shows nested dicts nicely formatted
```

**Why Jupyter-friendly:**
- While `pprint` works in regular Python too, Jupyter renders it beautifully
- Jupyter automatically formats dictionaries with collapsible sections
- Much easier to read complex nested structures in Jupyter

---

#### **📋 Quick Comparison: Jupyter vs Regular Python**

| Feature | Jupyter Notebook | Regular Python Script |
|---------|------------------|----------------------|
| Rich formatting | `display(Markdown("## Header"))` | `print("HEADER")` |
| Input boxes | `input()` creates text box | `input()` uses terminal |
| Output | Formatted, pretty, collapsible | Plain text only |
| Cell execution | One cell for entire workflow | Run as complete script |
| Visual feedback | Emojis, colors, markdown | Plain text |

---

#### **🔄 To Convert to Regular Python Script**

If you wanted to run this in a **terminal** instead of Jupyter:

```python
# REMOVE these Jupyter-specific imports:
# from IPython.display import display, Markdown

# REPLACE all display(Markdown(...)) with print():
# display(Markdown("## 🚀 PHASE 1..."))
print("\n" + "="*60)
print("🚀 PHASE 1: Starting agent...")
print("="*60 + "\n")

# Keep input() - it works in both!
choice = input("Enter your choice (1/2/3): ").strip()

# pprint works in both, but looks plainer in terminal
from pprint import pprint
pprint(response)
```

---

#### **✅ Summary**

**The 3 key Jupyter-specific elements:**

1. **`from IPython.display import display, Markdown`** - For rich output
2. **`display(Markdown("..."))`** - For pretty formatting  
3. **Single cell execution** - Keep everything together

**What works in BOTH:**
- `input()` - Creates text boxes in Jupyter, prompts in terminal
- `pprint()` - Works everywhere, just prettier in Jupyter
- All the LangChain/agent logic - 100% the same!

The Jupyter-specific parts are **only for making it look nice**. The actual HITL functionality (middleware, interrupts, resume) works the same everywhere! 🎯

## OK, let's now see the version of this code for Visual Studio Code

Here's the **terminal/Python script version** of the HITL code:

```python
"""
Human-in-the-Loop Email Agent - Terminal Version
Run this script from your terminal: python hitl_email_agent.py
"""

from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
from langgraph.types import Command
from pprint import pprint

# Tool definitions
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject."""
    return f"✅ Email sent successfully!\n\nBody: {body}"

# State and agent setup
class EmailState(AgentState):
    email: str

agent = create_agent(
    model="gpt-4o-mini",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

# Configuration
config = {"configurable": {"thread_id": "1"}}

# Helper function to print section headers
def print_header(text, symbol="="):
    """Print a formatted header"""
    print("\n" + symbol * 70)
    print(f"  {text}")
    print(symbol * 70 + "\n")

def print_subheader(text):
    """Print a formatted subheader"""
    print("\n" + "-" * 70)
    print(f"  {text}")
    print("-" * 70)

# Function to handle interrupts
def handle_interrupt(response, config, interrupt_number=1):
    """Handle a single interrupt and return the updated response"""
    
    if '__interrupt__' not in response:
        return response, False  # No interrupt, we're done
    
    print_header(f"⚠️  INTERRUPT #{interrupt_number} - Human approval required!", "!")
    
    # Extract interrupt info
    interrupt_obj = response['__interrupt__'][0]
    interrupt_value = interrupt_obj.value
    
    # Extract tool information from the interrupt value
    action_requests = interrupt_value.get('action_requests', [])
    if action_requests:
        action_request = action_requests[0]
        tool_name = action_request.get('name', 'unknown')
        tool_args = action_request.get('args', {})
        description = action_request.get('description', '')
        
        print(f"🤖 Agent wants to call: {tool_name}")
        print(f"\n📝 With arguments:")
        pprint(tool_args)
        
        if description:
            print(f"\n📄 Description:")
            print(description)
    else:
        print("⚠️  No action requests found in interrupt")
        return response, False
    
    # Get human input
    print_subheader(f"🤔 DECISION TIME (Interrupt #{interrupt_number})")
    print("\nWhat do you want to do?")
    print("  1 - Approve: Let the agent execute as planned")
    print("  2 - Reject: Stop the action and provide feedback")
    print("  3 - Edit: Modify the content before executing")
    
    choice = input("\n👉 Enter your choice (1/2/3): ").strip()
    
    # Resume based on decision
    print_subheader(f"⚡ RESUMING (Interrupt #{interrupt_number})...")
    
    if choice == "1":
        # APPROVE
        print("\n✅ You APPROVED the action\n")
        new_response = agent.invoke(
            Command(resume={"decisions": [{"type": "approve"}]}),
            config=config
        )
        return new_response, True
        
    elif choice == "2":
        # REJECT
        print("\n❌ You REJECTED the action")
        feedback = input("👉 Enter your feedback message: ").strip()
        print()
        new_response = agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {
                            "type": "reject",
                            "message": feedback
                        }
                    ]
                }
            ),
            config=config
        )
        return new_response, True
        
    elif choice == "3":
        # EDIT
        print("\n✏️  You chose to EDIT the action")
        new_body = input("👉 Enter the new email body: ").strip()
        print()
        new_response = agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {
                            "type": "edit",
                            "edited_action": {
                                "name": tool_name,
                                "args": {"body": new_body}
                            }
                        }
                    ]
                }
            ),
            config=config
        )
        return new_response, True
        
    else:
        print("\n❌ Invalid choice\n")
        return response, False


def main():
    """Main function to run the HITL agent"""
    
    # PHASE 1: Initial invocation
    print_header("🚀 PHASE 1: Starting Human-in-the-Loop Agent")
    
    print("Sending initial message to agent...")
    print("Email content: 'Hi, Donald. I will be in town tomorrow, will you have ")
    print("an available room in the White House? Best, Julio.'\n")
    
    response = agent.invoke(
        {
            "messages": [HumanMessage(content="Please read my email and send a response.")],
            "email": "Hi, Donald. I will be in town tomorrow, will you have an available room in the White House? Best, Julio."
        },
        config=config
    )
    
    # Handle all interrupts in a loop
    interrupt_count = 0
    max_interrupts = 5  # Safety limit to prevent infinite loops
    
    while '__interrupt__' in response and interrupt_count < max_interrupts:
        interrupt_count += 1
        response, continued = handle_interrupt(response, config, interrupt_count)
        
        if not continued:
            break
    
    # FINAL RESULT
    print_header("🎉 FINAL RESULT")
    
    if '__interrupt__' in response:
        print("⚠️  Still has pending interrupts (reached max limit or invalid choice)\n")
        print("Interrupt details:")
        pprint(response['__interrupt__'])
    else:
        print("✅ Process completed successfully!\n")
        
        # Show only the final messages
        if 'messages' in response:
            print("📧 Final Messages:")
            print("-" * 70)
            for msg in response['messages'][-3:]:  # Show last 3 messages
                msg_type = msg.type.upper() if hasattr(msg, 'type') else 'MESSAGE'
                msg_content = msg.content if hasattr(msg, 'content') else str(msg)
                print(f"\n[{msg_type}]")
                print(msg_content)
                print()


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️  Interrupted by user. Exiting...")
    except Exception as e:
        print(f"\n\n❌ Error occurred: {e}")
        import traceback
        traceback.print_exc()
```

---

## **Key Differences from Jupyter Version:**

#### **1. Removed Jupyter-Specific Imports**
```python
# REMOVED:
# from IPython.display import display, Markdown
```

#### **2. Replaced Markdown with Print Statements**
```python
# BEFORE (Jupyter):
display(Markdown("## 🚀 PHASE 1: Starting agent..."))

# AFTER (Terminal):
print_header("🚀 PHASE 1: Starting Human-in-the-Loop Agent")
```

#### **3. Added Helper Functions for Formatting**
```python
def print_header(text, symbol="="):
    """Print a formatted header"""
    print("\n" + symbol * 70)
    print(f"  {text}")
    print(symbol * 70 + "\n")
```
- Creates nice visual separators in the terminal
- Makes output more readable

#### **4. Wrapped Everything in main()**
```python
def main():
    """Main function to run the HITL agent"""
    # ... all the logic here ...

if __name__ == "__main__":
    main()
```
- Standard Python practice for scripts
- Makes it reusable and importable
- Cleaner structure

#### **5. Added Error Handling**
```python
try:
    main()
except KeyboardInterrupt:
    print("\n\n⚠️  Interrupted by user. Exiting...")
except Exception as e:
    print(f"\n\n❌ Error occurred: {e}")
```
- Handles Ctrl+C gracefully
- Shows errors properly in terminal

---

## **Example Terminal Output:**

```
======================================================================
  🚀 PHASE 1: Starting Human-in-the-Loop Agent
======================================================================

Sending initial message to agent...
Email content: 'Hi, Donald. I will be in town tomorrow, will you have 
an available room in the White House? Best, Julio.'

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
  ⚠️  INTERRUPT #1 - Human approval required!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

🤖 Agent wants to call: send_email

📝 With arguments:
{'body': 'Hi Julio,\n\nYes, we will have a room available...'}

----------------------------------------------------------------------
  🤔 DECISION TIME (Interrupt #1)
----------------------------------------------------------------------

What do you want to do?
  1 - Approve: Let the agent execute as planned
  2 - Reject: Stop the action and provide feedback
  3 - Edit: Modify the content before executing

👉 Enter your choice (1/2/3): 1

----------------------------------------------------------------------
  ⚡ RESUMING (Interrupt #1)...
----------------------------------------------------------------------

✅ You APPROVED the action

======================================================================
  🎉 FINAL RESULT
======================================================================

✅ Process completed successfully!

📧 Final Messages:
----------------------------------------------------------------------

[AI]
...

[TOOL]
✅ Email sent successfully!

Body: Hi Julio,...
```

---

## Benefits of This Terminal Version

✅ **Runs anywhere** - No Jupyter needed
✅ **Cleaner structure** - Uses functions and main()
✅ **Better error handling** - Catches Ctrl+C and exceptions
✅ **Professional formatting** - Nice headers and separators
✅ **Easy to deploy** - Can run on servers, CI/CD, etc.
✅ **Reusable** - Can import functions into other scripts

This version does **exactly the same thing** as the Jupyter version, just optimized for terminal execution! 🚀

## How to run this code from Visual Studio Code
* Open Terminal.
* Make sure you are in the project folder.
* Make sure you have the poetry env activated.
* Enter and run the following command:
    * `python 013-mid-to-add-HITL.py`